# CSV Merger (base-name merge)

This notebook merges `exp2_rune.csv` and `exp2_fred.csv` by the base image id (e.g. `syn_img_5`). It computes average ratings across both raters and both files.

In [ ]:
import pandas as pd

def base(name):
    # Safely extract the numeric suffix before the extension, e.g., 1.83 from syn_img_5_1.83.jpg
    if not isinstance(name, str):
        return ''
    name_no_ext = name.rsplit('.', 1)[0]
    parts = name_no_ext.split('_')
    return parts[-1] if parts else ''

# Read input CSVs
df1 = pd.read_csv('exp2_rune.csv')
df2 = pd.read_csv('exp2_fred.csv')

# Add base id column
df1['Base'] = df1['Image'].map(base)
df2['Base'] = df2['Image'].map(base)

# Convert to long format (one rating per row)
df1_long = df1.melt(id_vars=['Base'], value_vars=['Rating 1', 'Rating 2'], value_name='rating').drop(columns=['variable']).rename(columns={'value':'rating'})
df2_long = df2.melt(id_vars=['Base'], value_vars=['Rating 1', 'Rating 2'], value_name='rating').drop(columns=['variable']).rename(columns={'value':'rating'})

# Combine and compute mean rating per Base
combined = pd.concat([df1_long[['Base','rating']], df2_long[['Base','rating']]], ignore_index=True)
result = combined.groupby('Base', sort=False)['rating'].mean().reset_index().rename(columns={'rating':'average_rating'})

# Save outputs
result.to_csv('average_ratings_by_base.csv', index=False)
combined.to_csv('ratings_long.csv', index=False)

print(f'Computed average ratings for {len(result)} base images.')
print(result.head(10).to_string(index=False))

Computed average ratings for 0 base images.
Empty DataFrame
Columns: [Base, average_rating]
Index: []
